# Harmony4D Drive Benchmark (Colab)

Chạy `run_drive_benchmark.py` trên Google Colab (tự tải từ Drive ID, chấm cả `aria01` + `aria02`, backup CSV sang Google Drive mỗi phút để không mất tiến độ khi ngắt runtime).

In [ ]:
#@title 1. Cài môi trường, clone repo và tải SMPL_NEUTRAL.pkl
import os, re, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/doantrunghieu08/optimization_model_monocular_3.git"  #@param {type:"string"}
BRANCH = "experiment/observability-aware-trust-region"  #@param {type:"string"}
SMPL_NEUTRAL_FILE_ID = "1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll"  #@param {type:"string"}
FUSION_METHOD = "aotr"  #@param ["aotr", "partwise", "naive"]
CONFIGS = ""  #@param {type:"string"}
REVERSE = True  #@param {type:"boolean"}
WORKERS = 0  #@param {type:"integer"}
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/optimization_model_monocular_3/drive_benchmark"  #@param {type:"string"}

REPO_DIR = Path("/content/optimization_model_monocular_3")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-f", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "chumpy"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "PyYAML", "numpy<2.3"], check=True)

smpl_path = REPO_DIR / "models" / "SMPL_NEUTRAL.pkl"
if not smpl_path.exists() or smpl_path.stat().st_size < 1_000_000:
    import gdown
    smpl_path.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=SMPL_NEUTRAL_FILE_ID, output=str(smpl_path), quiet=False)

cfg_path = REPO_DIR / "configs" / "pipeline.yml"
cfg_text = re.sub(r"(?m)^(\s*method:\s*).*$", rf"\g<1>{FUSION_METHOD}", cfg_path.read_text(encoding="utf-8"), count=1)
cfg_path.write_text(cfg_text, encoding="utf-8")
print(f"Ready | branch={BRANCH} | method={FUSION_METHOD} | cpus={os.cpu_count()}")

In [ ]:
#@title 2. Chạy benchmark (tự sync CSV sang Drive để resume khi chạy lại)
import os, shutil, subprocess, sys, threading
from pathlib import Path
import yaml
from run_drive_benchmark import is_drive_segment

backup_root = None
out_root = REPO_DIR / "output" / "drive_benchmark"
if DRIVE_BACKUP_DIR.strip():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    backup_root = Path(DRIVE_BACKUP_DIR.strip())
    backup_root.mkdir(parents=True, exist_ok=True)
    for src in backup_root.glob("*/brute_force_report_*.csv"):
        dst = out_root / src.parent.name / src.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)

def sync_csvs(seq_name):
    if backup_root is None:
        return
    for csv_file in (out_root / seq_name).glob("brute_force_report_*.csv"):
        dst = backup_root / seq_name / csv_file.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(csv_file, dst)

workers = WORKERS if WORKERS > 0 else max(1, (os.cpu_count() or 2) - 1)
targets = [REPO_DIR / c for c in CONFIGS.split()] if CONFIGS.strip() else sorted((REPO_DIR / "configs").glob("*.yml"))
targets = [p for p in targets if any(is_drive_segment(s) for s in (yaml.safe_load(p.read_text(encoding="utf-8")) or {}).get("segments", []))]
if REVERSE:
    targets = list(reversed(targets))

env = {**os.environ, "PYTHONUTF8": "1", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"}
for cfg in targets:
    cmd = [sys.executable, "-u", "run_drive_benchmark.py", "--workers", str(workers), *(["--reverse"] if REVERSE else []), str(cfg)]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stop_sync = threading.Event()
    def _sync_loop(seq=cfg.stem):
        while not stop_sync.wait(60):
            sync_csvs(seq)
    t = threading.Thread(target=_sync_loop, daemon=True)
    t.start()
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    stop_sync.set()
    t.join()
    sync_csvs(cfg.stem)
    if proc.returncode not in (0, 1):
        raise RuntimeError(f"{cfg.name} failed with exit code {proc.returncode}")

In [ ]:
#@title 3. Tổng hợp nhanh kết quả đã chạy
import glob, pandas as pd

rows = []
for path in sorted(glob.glob(str(REPO_DIR / "output/drive_benchmark/*/brute_force_report_*.csv"))):
    df = pd.read_csv(path)
    df = df[df["Protocol"].astype(str) != "End"]
    for (seg, person), g in df.groupby(["Segment", "Target Person"]):
        rows.append({
            "Segment": seg, "Person": person, "Pairs": len(g),
            "PA mono": round(pd.to_numeric(g["pa-mpjpe.all_baseline"]).mean(), 2),
            "PA fused": round(pd.to_numeric(g["pa-mpjpe.all_method"]).mean(), 2),
            "MPJPE mono": round(pd.to_numeric(g["mpjpe.all_baseline"]).mean(), 2),
            "MPJPE fused": round(pd.to_numeric(g["mpjpe.all_method"]).mean(), 2),
        })
pd.DataFrame(rows)